# Part 4h — Joint ISOMAP: A Unified LOB Manifold Across NVDA, AAPL, MSFT

**Goal:** Fit *one* ISOMAP on pooled LOB data from all three symbols and ask:
- Do NVDA, AAPL, and MSFT occupy separate regions of the shared manifold, or do they overlap?
- Does the Z₁/Z₂ two-axis interpretation survive joint training?
- How does OOS prediction compare to per-symbol models?

**Pipeline:**
1. Build 1-min OBI bars for each symbol (same Oct 2–31 2023 window, ~8,579 bars each)
2. Pool all three into one ~19,300-bar training set (no scaler — OBI already in [−1, 1])
3. Fit ONE Isomap on the pooled data
4. Color-code points by symbol and time-of-day
5. Check depth profile: does Z₁/Z₂ interpretation survive?
6. UMAP for cluster detection
7. OOS projection and IC comparison

In [ ]:
from IPython.display import Image, display
import subprocess, sys

# Run the joint pipeline (takes ~3-5 min for UMAP on 19k points)
# Uncomment to re-generate figures:
# subprocess.run([sys.executable, 'run_4h_joint_isomap.py'], check=True)

## Why No Within-Symbol Scaler?

OBI is defined as $(B_k - A_k) / (B_k + A_k)$, so it is **bounded in [−1, 1] by construction** for every symbol and every depth level. There is no scale difference to remove. Applying `StandardScaler` before ISOMAP inflates the reconstruction error above 1 (worse than baseline) — the same pathology seen in Part 4f. Raw OBI values are the correct input.

If we were pooling features with different units (e.g., OBI together with dollar-quoted spread), normalization would be required. Here it is not.

## Step 1: Joint Embedding — Do Symbols Overlap?

The left panel colors each point by symbol; the right panel colors by time of day.

In [ ]:
display(Image('figures/4h_joint_embedding.png'))

**What we see:**

**Left — by symbol:** NVDA, AAPL, and MSFT points are substantially **overlapping** on the shared manifold. There is no clean separation between stocks. This tells us the LOB geometry is driven by *market-wide microstructure regimes* (open, mid-day, close) rather than stock-specific signals.

**Right — by time of day:** The familiar open/close cluster at one end and the calmer mid-day region at the other reappears — now pooled across all three symbols. The time-of-day structure is **stronger** than the stock-identity structure.

**Takeaway:** The LOB manifold is approximately *universal* across large-cap Nasdaq names in the same market session. Stock identity is a second-order effect.

## Step 2: Reconstruction Quality

| Metric | Value |
|--------|-------|
| ISOMAP reconstruction error | 0.0273 → **97.3% geodesic variance preserved** |
| PCA 2D variance explained | 81.9% |
| ISOMAP gap over PCA | **+15.4 pp** |

The joint model preserves 97.3% of the pooled manifold's geodesic structure in just 2 dimensions — matching the per-symbol fits (NVDA 97.2%, AAPL 96.7%, MSFT 96.7%). Pooling three symbols did not degrade embedding quality.

## Step 3: Depth Profile — Does Z₁/Z₂ Interpretation Survive?

The depth profile shows Spearman ρ between each ISOMAP coordinate and each of the 10 raw OBI levels.

In [ ]:
display(Image('figures/4h_joint_depth_profile.png'))

**Reading the chart:**

| Axis | Pattern | Economic Meaning |
|------|---------|------------------|
| **Z₁** | Same sign across all 10 levels; peaks at L5 (ρ = +0.894) | **Book-wide consensus** — how bullish/bearish is the entire stack? |
| **Z₂** | Positive at L0–L4, crosses zero at L5, negative at L6–L9 | **HFT-vs-institutional contrast** — shallow HFT pressure vs deep institutional flow |

This is **identical** to the per-symbol structure found in Parts 4f and 4g, modulo the arbitrary sign of Z₂ (the axis is reflected relative to some per-symbol fits, but the sign-flip pattern is the same).

**The Z₁/Z₂ two-axis interpretation is robust to joint training.** It is not a per-symbol artifact — it is a structural property of LOB geometry.

## Step 4: OOS Projection onto the Joint Manifold

In [ ]:
display(Image('figures/4h_joint_oos.png'))

OOS bars from all three symbols (last 25% of October, ~2,145 bars per symbol) project cleanly inside the training region. No symbol drifts outside the learned manifold boundary. This validates the Nyström extension (nearest-neighbor approximate projection) for multi-symbol OOS data.

## Step 5: Joint UMAP — Regime Clusters

In [ ]:
display(Image('figures/4h_joint_umap.png'))

**Left — K-Means regimes:** K=2 is again optimal (silhouette = 0.538, highest across all three symbols and both per-symbol and joint models). The two regimes align with the open/close vs. mid-day partition identified in earlier parts.

**Right — colored by symbol:** Within each UMAP cluster, all three symbols are mixed together. Stock identity does not determine cluster membership — confirming that regime structure is market-wide, not stock-specific.

**K=2 universality across all models:**

| Model | Best K | Silhouette |
|-------|--------|------------|
| NVDA per-symbol | 2 | 0.491 |
| AAPL per-symbol | 2 | 0.494 |
| MSFT per-symbol | 2 | 0.511 |
| **Joint (pooled)** | **2** | **0.538** |

The joint model has the *highest* silhouette — pooling sharpens the cluster structure.

## Step 6: OOS IC Comparison

In [ ]:
display(Image('figures/4h_joint_oos_ic.png'))

**OOS IC results (pooled cross-symbol):**

| Feature set | OOS IC | t-stat | Significance |
|-------------|--------|--------|--------------|
| Raw OBI (10D) | +0.0192 | +1.54 | n.s. |
| **Joint ISOMAP 2D** | −0.0030 | −0.24 | n.s. |
| Joint UMAP 2D | +0.0179 | +1.44 | n.s. |
| Joint PCA 2D | +0.0026 | +0.21 | n.s. |

All ICs are near zero in the pooled cross-symbol setting. This is **expected**: predicting 1-minute returns for a mixture of three large-cap stocks from LOB state alone is extremely hard. The geometry captured by the joint model is meaningful (97.3% preservation, clean regimes) — it just does not directly translate into short-horizon return prediction in a pooled setting without per-symbol calibration.

The per-symbol ICs from Part 4g were small but more consistent (+0.004 to +0.020), because the model learned the specific signal structure of each stock separately.

## Summary

| Finding | Result |
|---------|--------|
| Symbol separation on manifold | **Overlap** — stocks share the same manifold |
| Reconstruction error (joint) | **0.0273** → 97.3% preserved |
| PCA 2D baseline | 81.9% |
| ISOMAP advantage over PCA | +15.4 pp |
| Z₁/Z₂ interpretation | **Survives** joint training — same consensus/contrast axes |
| Optimal clusters (joint UMAP) | **K=2**, silhouette 0.538 (strongest yet) |
| Cluster composition | Mixed symbols — regimes are market-wide |
| OOS IC (pooled prediction) | Near zero for all methods — calibration needed per-symbol |

**Key conclusion:** The LOB manifold geometry is approximately universal across large-cap Nasdaq names in the same market window. The two structural axes (Z₁=book consensus, Z₂=HFT-vs-institutional contrast) emerge regardless of whether we fit ISOMAP on one stock or all three simultaneously. Market microstructure regimes are driven by common intraday patterns — open/close activity bursts vs. mid-day calm — rather than by stock-specific factors.

**Next step (Part 4i):** Project the stress period (Aug 2024 BOJ shock week, when NVDA fell ~\$30) onto the *calm* manifold trained here. We expect stress-period points to fall in unusual regions — an unsupervised early-warning signal for regime change.